In [1]:
import pandas as pd

In [2]:
from indicateurs import TVAIndicators,ControleIndicators
from core.eligible_builder import EligibleBuilder 
from core.risk_compute import RiskComputer

In [3]:
dcf_prog =pd.read_csv("D:/DGI/project/anarisk-r/data/DCF_PROG_05_01_2026.csv",encoding='latin1' )

In [5]:
dcf_prog.head()

,ID_CONTR,NUM_IFU,NOM.x,ENCOMMERC,NOM_MINEFID,CREATEUR,DATE_CREATION,MODIFICATEUR,DATE_MODIFICATION,ETAT,...,DERNIERE_GESTION_SOUMIS_VERIF,CA_HTVA,BENEFICE_IMPOSABLE,IBENEF_EXIGIBLE,IBENEF_DUES,PREL_SOURCE_ACOMPTE,RETENUE_SOURCE_IMPUTE,TÃ©lÃ©phone,Groupe_IFU,Occurrences_IFU
0,481483.0,00026339U,GILBERT KIENDREBEOGO,NaN,GILBERT KIENDREBEOGO,ACHILZ,2010-04-23T00:00:00Z,SINTAX,2023-06-03T00:00:00Z,ACTIF,...,2023.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,481129.0,00026269T,FOLIDIA DIT JOEL TANKOANO,LYCEE PRIVE EMMAUS,FOLIDIA DIT JOEL TANKOANO (LYCEE PRIVE EMMAUS),COULIBALYL,2010-04-21T00:00:00Z,SINTAX,2023-06-03T00:00:00Z,ACTIF,...,2023.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,481132.0,00026274N,PINSAPO GOLD,PINSAPO GOLD SA,PINSAPO GOLD (PINSAPO GOLD SA),KABORES,2010-04-21T00:00:00Z,SINTAX,2024-09-14T00:00:00Z,ACTIF,...,2024.0,0.0,-6145000.0,1000000.0,1000000.0,0.0,0.0,NaN,NaN,NaN
3,481132.0,00026274N,PINSAPO GOLD,PINSAPO GOLD SA,PINSAPO GOLD (PINSAPO GOLD SA),KABORES,2010-04-21T00:00:00Z,SINTAX,2024-09-14T00:00:00Z,ACTIF,...,2024.0,0.0,-5552412.0,1000000.0,250000.0,0.0,0.0,NaN,NaN,NaN
4,481132.0,00026274N,PINSAPO GOLD,PINSAPO GOLD SA,PINSAPO GOLD (PINSAPO GOLD SA),KABORES,2010-04-21T00:00:00Z,SINTAX,2024-09-14T00:00:00Z,ACTIF,...,2024.0,0.0,0.0,1000000.0,250000.0,0.0,0.0,NaN,NaN,NaN


In [4]:
dcf_prog.duplicated().sum()

np.int64(138)

In [3]:
q2 =pd.read_excel("../data/Q2.xlsx")

In [4]:
q2.columns

Index(['STRUCTURES ', 'BRIGADES', 'NUM_IFU', 'NOM', 'ACTIVITES', 'PROGRAMME',
       'ANNEE_ GESTION', 'TYPE_ CONTROLE'],
      dtype='object')

In [7]:
q2["BRIGADES"].to_list()

['BV1-DME-C3',
 'BV2-DME-C4',
 'BV3-DME-C4',
 'BV4-DME-C4',
 'BV4-DME-C4',
 'BV4-DME-C4',
 'BV4-DME-C4',
 'BV4-DME-C4',
 'BV-OUAGA2',
 'BV-OUAGA2',
 'BV-OUAGA2',
 'BV-OUAGA3',
 'BV-OUAGA4',
 'BV-OUAGA8',
 'BV-OUAGA9',
 'BV-OUAGA9',
 'BV-OUAGA9',
 'BV-DRI-CS',
 'BV-DRI-CSC',
 'BV-BOBO2',
 'BV-DRI-SO',
 'BV2-DME-CI',
 'BV-OUAGAVII',
 'BV-OUAGAVII']

In [6]:
insd_data = pd.read_excel(f"../data/BASE_DONNEES_INSD.xlsx")

In [9]:
insd_data["Num_IFU_Contribuable"]

0       00121898C
1       00040202C
2       00134964A
3       00036488G
4       00008986R
          ...    
7636    00107467S
7637    00069136C
7638    00037460R
7639    00024404E
7640    00071156X
Name: Num_IFU_Contribuable, Length: 7641, dtype: object

In [ ]:
dcf_prog["NUM_IFU"].nunique()

0    00026339U
1    00026269T
2    00026274N
3    00026274N
4    00026274N
Name: NUM_IFU, dtype: object

In [42]:
dcf_prog[(dcf_prog["ETAT"]=="ACTIF") & (dcf_prog["CODE_REG_FISC"]=="RSI")].drop_duplicates()["NUM_IFU"].nunique() 

17873

In [84]:
mask_nul = dcf_prog["NUM_IFU"].isin([None, np.nan, pd.NaT, '', ' ', 'NULL', 'null', 'None'])
print(f"Valeurs considérées comme nulles: {mask_nul.sum()}")

Valeurs considérées comme nulles: 186


In [85]:
# Vérifier si la colonne a des types mixtes
print("=== TYPES DE DONNÉES ===")
types_uniques = dcf_prog["NUM_IFU"].apply(type).unique()
print(f"Types uniques dans la colonne: {types_uniques}")

# Convertir tout en string pour comparaison
dcf_prog["NUM_IFU_STR"] = dcf_prog["NUM_IFU"].astype(str)
print(f"Après conversion en string: {dcf_prog['NUM_IFU_STR'].nunique()}")

=== TYPES DE DONNÉES ===
Types uniques dans la colonne: [<class 'str'> <class 'float'>]
Après conversion en string: 613947


In [28]:
dcf_prog.drop_duplicates(inplace=True)

In [29]:
dcf_prog.shape

(659671, 170)

In [21]:
r = dcf_prog[(dcf_prog["DATE_DERNIERE_VG"]<"2023-12-31") & 
            (dcf_prog["DATE_DERNIERE_AVIS"]<"2023-12-31") &
            (dcf_prog["DATE_DERNIERE_VP"]<"2023-12-31")]


In [22]:
print(f"r shape: {r.shape} ")

r shape: (390726, 170) 


In [24]:
r["DATE_DERNIERE_VG"] =pd.to_datetime(r["DATE_DERNIERE_VG"], errors='coerce')

In [27]:
r["DATE_DERNIERE_VG"].dt.year.value_counts().sort_values()

DATE_DERNIERE_VG
1981.0        1
1996.0        1
1992.0        1
1954.0        1
1950.0        1
          ...  
2022.0    37333
2018.0    39708
2019.0    42211
2020.0    47271
2021.0    48948
Name: count, Length: 61, dtype: int64

In [30]:
r[r["DATE_DERNIERE_VG"].dt.year < 2000]

,ID_CONTR,NUM_IFU,NOM.x,ENCOMMERC,NOM_MINEFID,CREATEUR,DATE_CREATION,MODIFICATEUR,DATE_MODIFICATION,ETAT,...,DERNIERE_GESTION_SOUMIS_VERIF,CA_HTVA,BENEFICE_IMPOSABLE,IBENEF_EXIGIBLE,IBENEF_DUES,PREL_SOURCE_ACOMPTE,RETENUE_SOURCE_IMPUTE,TÃ©lÃ©phone,Groupe_IFU,Occurrences_IFU
124505,618248.0,RF1604524,SAWADOGO Somyalma,NaN,SAWADOGO Somyalma,SOUGOUM,2016-03-10T00:00:00Z,SOUGOUM,2016-03-10T00:00:00Z,ACTIF,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
124590,618252.0,RF1604528,SAWADOGO Somyalma,NaN,SAWADOGO Somyalma,SOUGOUM,2016-03-10T00:00:00Z,SOUGOUM,2016-03-10T00:00:00Z,ACTIF,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
131451,614368.0,RF1602072,NOAGO Bourgou dit Boukari,tradi-praticien,NOAGO Bourgou dit Boukari (tradi-praticien),SOUGOUM,2016-02-04T00:00:00Z,SOUGOUM,2016-02-04T00:00:00Z,ACTIF,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
296537,828017.0,RF2101347,OUEDRAOGO INNOCENT,NaN,OUEDRAOGO INNOCENT,AKARA,2021-01-19T00:00:00Z,AKARA,2021-01-19T00:00:00Z,ACTIF,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
419544,943073.0,RF2301265,KIEMDE B ZACHARI,NaN,KIEMDE B ZACHARI,CPARLETTE,2023-01-17T00:00:00Z,CPARLETTE,2023-01-17T00:00:00Z,ACTIF,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
644123,1191606.0,RF2563639,SAWADOGO MOUMOUNI,NaN,SAWADOGO MOUMOUNI,DKARIMOU,2025-09-30T14:58:21Z,DKARIMOU,2025-09-30T14:58:21Z,ACTIF,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
645211,1175334.0,RF2553558,KAMBIRE SIE N°1,NaN,KAMBIRE SIE N°1,DKARIMOU,2025-08-18T09:31:02Z,DKARIMOU,2025-08-18T09:31:02Z,ACTIF,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
645297,1185935.0,RF2560100,TRAORE MAHAMED,NaN,TRAORE MAHAMED,DKARIMOU,2025-09-16T14:14:59Z,DKARIMOU,2025-09-16T14:14:59Z,ACTIF,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
645575,1200286.0,RF2569233,KAMBIRE ENOCK,NaN,KAMBIRE ENOCK,DKARIMOU,2025-10-23T13:57:01Z,DKARIMOU,2025-10-23T13:57:01Z,ACTIF,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
rsi = r[r["CODE_REG_FISC"]=="RSI"]

In [19]:
rsi.shape

(13199, 170)

In [39]:
risk_computer = RiskComputer()

2026-02-06 10:22:28,100 [core.risk_compute] Chargement des données de référence...
2026-02-06 10:22:29,028 [core.risk_compute] NON_EIGIBLE.xlsx chargé: 6984 IFU a exclur
2026-02-06 10:22:29,063 [core.risk_compute] Entreprises_LIEES.xlsx chargé: 100 lignes


In [40]:
data_eligible = risk_computer.exclude_non_eligible(r)

2026-02-06 10:22:46,839 [core.risk_compute] Exclusion non-éligibles: 7260 contribuables exclus sur 349478


In [42]:
data_eligible.shape

(342218, 170)

In [45]:
r.columns = r.columns.str.upper()

In [43]:
print(r.shape[0]-data_eligible.shape[0])

7260


In [46]:
mtn_tva_net= r["MONTANT_TVA_NET_A_PAYER_25"]
mnt_tva_deduction = r["FOURN_TVA_DEDUCTIBLE_AN"]

In [47]:
import numpy as np

mtn_tva_net == -2
mnt_tva_deduction = -1


In [ ]:
ratio = np.select([mnt_tva_deduction==0,mtn_tva_net==0], [-1,-2], default=mtn_tva_net/mnt_tva_deduction)

In [70]:
len(ratio[(ratio>0) & (ratio<0.20)])

556

In [ ]:
risk_ind = np.select([ratio>0.20,ratio==-1,ratio==-2], ["vert","mnt_tva_deduction null","mtn_tva_net null"], default="non disponible")

In [ ]:

criticite =5
def calc_ecart(ratio,mnt_tva_deduction,mnt_tva_net):
    if ratio == "vert":
        return {"GAP_IND_1": 0, "SCORE_IND_1": 0, "RISQUE_IND_1": "vert"}
    else :
        ecart =  np.abs((0.20*mnt_tva_deduction) - mnt_tva_net)*0.80
        impact = np.select([ecart<500000,ecart<5000000,ecart<20000000,100000000], [1,2,3,4,5], default=0)
        scores = impact*criticite
        RISQUE_IND_1 = np.select([np.isin(scores, [1, 2, 3, 4]),
                                scores == 5,
                                scores == 6,
                                np.isin(scores, [8, 9]),
                                scores == 10,
                                scores == 12,
                                scores == 15,
                                scores == 16,
                                np.isin(scores, [20, 25]), 
                                ], [ "vert", "vert","vert","jaune", "jaune","orange","orange", "orange", "rouge"], default="non disponible")
        RISQUE_IND_1[RISQUE_IND_1]
        mask_score5 = scores == 5
        RISQUE_IND_1[mask_score5] = np.where(criticite[mask_score5] == 1, "vert", "jaune")
        mask_score6 = scores == 6
        RISQUE_IND_1[mask_score6] = np.where(criticite[mask_score6] == 2, "vert", "jaune")
        mask_score10 = scores == 10
        RISQUE_IND_1[mask_score10] = np.where(criticite[mask_score10] == 2, "jaune", "rouge")
        mask_score15 = scores == 15
        RISQUE_IND_1[mask_score15] = np.where(criticite[mask_score15] == 3, "orange", "rouge")

        return {"GAP_IND_1": ecart, "SCORE_IND_1": scores, "RISQUE_IND_1": RISQUE_IND_1}



vectorized_calc_ecart = np.vectorize(calc_ecart)

In [75]:
r[["FOURN_TVA_DEDUCTIBLE_AN", "MONTANT_TVA_NET_A_PAYER_25"]].columns

Index(['FOURN_TVA_DEDUCTIBLE_AN', 'MONTANT_TVA_NET_A_PAYER_25'], dtype='object')

In [78]:
tva = TVAIndicators()

In [ ]:
tva.calculate_indicator_1()